In [9]:
import pandas as pd, altair as alt
from ecostyles import EcoStyles            # adjust import
styles = EcoStyles(); styles.register_and_enable_theme()

FS = None                                      # set e.g. 11 to shrink ALL chart text (axes, key, caption); None = theme default

# World Bank, International Debt Statistics 2026 -- Low & middle income, $ millions
years = list(range(2015, 2026))
lines = {
    "External debt": [5915711.00, 6136354.20, 6933133.60, 7382345.10, 7757739.10, 8182110.40,
                      8829210.50, 8726307.90, 8889597.70, 8949585.50, 9445017.60],      # Total external debt stocks
    "Interest payments": [142681.90, 150677.20, 167974.00, 191904.20, 211894.30, 184936.90,
                          204583.00, 200884.30, 257347.00, 268152.10, 275758.40],        # Interest payments (long-term)
    "National income": [24754589.80, 25027172.00, 27380646.50, 29064676.60, 30011997.60, 29077827.10,
                        34287307.60, 35831117.40, 36926837.20, 38212767.10, 40163857.10],  # Gross national income (GNI)
}
idx = pd.concat({k: 100 * pd.Series(v, index=years) / v[0] for k, v in lines.items()},
                names=["series", "year"]).rename("value").reset_index()
last = idx[idx.year == years[-1]].set_index("series").value
print(last.round(1).to_dict())                 # should read debt ~160, income ~162, interest ~193

# all three series take their colours from the theme palette, in order; the key sits at the bottom
SERIES = ["External debt", "Interest payments", "National income"]
col = alt.Color("series:N", scale=alt.Scale(domain=SERIES),
                legend=alt.Legend(orient="bottom", direction="horizontal", title=None,
                                  symbolType="stroke", symbolOpacity=1, symbolStrokeWidth=3))
XS = alt.Scale(domain=[years[0], years[-1]], nice=False); YS = alt.Scale(domain=[100, 200], nice=False)
X = alt.X("year:Q", title=None, scale=XS,
          axis=alt.Axis(values=list(range(years[0], years[-1] + 1, 2)), format="d", grid=False))
Y = alt.Y("value:Q", title="2015 = 100", scale=YS,
          axis=alt.Axis(values=[100, 125, 150, 175, 200], format="d", titleAngle=0, titleAlign="left", titleY=-10))

income = alt.Chart(idx[idx.series == "National income"]).mark_line(strokeWidth=1.8).encode(X, Y, color=col)
main = alt.Chart(idx[idx.series != "National income"]).mark_line(strokeWidth=1.8).encode(X, Y, color=col)

chart = (income + main).properties(
    width=400, height=300)

styles.save(chart, path=".", name="ids_external_debt", svg=True)

{'External debt': 159.7, 'Interest payments': 193.3, 'National income': 162.2}
